# Gemma 4 Local Server on Google Colab (for `legal-copilot-backend`)

এই নোটবুকটি আপনার **বিদ্যমান Gemini + Groq সিস্টেম একদম অক্ষুণ্ণ রেখে** Google Colab GPU-তে স্থানীয়ভাবে **Gemma 4** চালাবে এবং একটি সুরক্ষিত (Bearer token protected) OpenAI-compatible URL দেবে।

### GPU অনুযায়ী মডেল পছন্দ:
- **Colab Free (T4 15GB VRAM):** `MODEL_NAME = "gemma4:e4b"` (দ্রুত ও নিরাপদে চলবে)
- **Colab Pro (L4 24GB / A100 40GB+):** `MODEL_NAME = "gemma4:26b"`

### কীভাবে চালাবেন:
1. Colab মেনু থেকে **Runtime -> Change runtime type -> T4 GPU** (অথবা L4/A100) সিলেক্ট করুন (**CPU হলে Timeout হবে!**)।
2. নিচের সেলগুলো ১ থেকে ৪ পর্যন্ত ক্রমান্বয়ে রান করুন।
3. শেষ সেলে প্রিন্ট হওয়া `.env` লাইনগুলো কপি করে আপনার লোকাল `legal-copilot-backend/.env`-এ পেস্ট করে সার্ভার রিস্টার্ট করুন।
4. টেস্ট শেষে শুধু `.env`-এ `COLAB_LLM_BASE_URL` লাইনটি কমেন্ট (`#`) করে দিলেই পুরো সিস্টেম আবার আগের Gemini + Groq মোডে ফিরে যাবে।

In [ ]:
!nvidia-smi
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i /tmp/cloudflared.deb

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

# Verify GPU first — CPU runtime will be too slow and cause timeouts
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU and restart."
    )
print("\n".join(gpu_check.stdout.splitlines()[:10]))

# Free Colab T4 (15GB VRAM) — 3 local benchmark models (1 loaded at a time to prevent VRAM OOM)
MODELS_TO_BENCHMARK = ["gemma4:e4b", "qwen3:8b", "llama3.1:8b"]
MODEL_NAME = MODELS_TO_BENCHMARK[0]
NUM_CTX = 12288  # Fits long 17-clause contracts + Playbook + Qdrant statutes cleanly on T4 15GB
THINK_MODE = False  # False = fast & reliable strict JSON output for legal-copilot-backend

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "15m"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_NUM_GPU"] = "999"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"

def _ollama_ready():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

# Restart ollama serve cleanly so GPU/env vars above are always applied
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(30):
    if _ollama_ready():
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start. Inspect /tmp/ollama.log")

print("Ollama server is running on 127.0.0.1:11434")
for m in MODELS_TO_BENCHMARK:
    print(f"Pulling {m}...")
    subprocess.run(["ollama", "pull", m], check=True)

print("\n=== Installed Ollama Models ===")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

# Warm up model into VRAM and verify 100% GPU placement
print("Warming up model into GPU VRAM (first load can take 30-90s)...")
warmup_req = urllib.request.Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps({
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say OK"}],
        "stream": False,
        "think": THINK_MODE,
        "options": {"num_predict": 8, "num_ctx": NUM_CTX},
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(warmup_req, timeout=300) as resp:
    warmup_res = json.loads(resp.read().decode("utf-8"))
print("Warmup reply:", warmup_res.get("message", {}).get("content", "").strip())

ps_out = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
print("\n=== ollama ps (PROCESSOR should say 100% GPU) ===")
print(ps_out.stdout)

In [ ]:
# Quick local Colab sanity check with streaming (Bangla, English, Banglish legal questions)
import json
import time
import urllib.request

sample_queries = [
    ("Bangla", "চেক ডিজঅনার হলে Negotiable Instruments Act 1881-এর ধারা ১৩৮ অনুযায়ী মামলা করার সময়সীমা কী?"),
    ("English", "Under the Contract Act, 1872 in Bangladesh, is a 2-year post-employment non-compete enforceable? Cite the section."),
    ("Banglish", "Amar malik notice chara chakri theke ber kore dise, Bangladesh Labour Act onujayi amar ki adhikar ace? Bangla script e bolo."),
]

for lang, q in sample_queries:
    t0 = time.perf_counter()
    print(f"=== [{lang}] ===", flush=True)
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps({
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": "You are a Bangladesh-law legal assistant. Reply concisely in Bangla script for Bangla/Banglish queries and English for English queries."},
                {"role": "user", "content": q},
            ],
            "stream": True,
            "think": THINK_MODE,
            "options": {"temperature": 0.1, "num_predict": 500, "num_ctx": NUM_CTX},
        }).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        for raw_line in resp:
            chunk = json.loads(raw_line.decode("utf-8"))
            piece = (chunk.get("message") or {}).get("content", "")
            if piece:
                print(piece, end="", flush=True)
    dt = round(time.perf_counter() - t0, 2)
    print(f"\n-- ({dt}s) --\n", flush=True)

In [ ]:
# Authenticated OpenAI-compatible proxy (port 11435) + Cloudflare Quick Tunnel
import json
import re
import secrets
import subprocess
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

API_TOKEN = secrets.token_urlsafe(24)
PROXY_PORT = 11435

# Clean up any previous proxy/tunnel if this cell is re-run
if "proxy_server" in globals() and proxy_server is not None:
    try:
        proxy_server.shutdown()
        proxy_server.server_close()
    except Exception:
        pass
subprocess.run(["pkill", "-f", "cloudflared"], check=False)


class ReusableThreadingHTTPServer(ThreadingHTTPServer):
    allow_reuse_address = True


class ColabOpenAIProxyHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # keep Colab output clean

    def _send_json(self, code, payload):
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/health":
            self._send_json(200, {"status": "ok", "default_model": MODEL_NAME, "models": MODELS_TO_BENCHMARK, "think": THINK_MODE})
            return
        self._send_json(404, {"error": "Not found"})

    def do_POST(self):
        auth = self.headers.get("Authorization", "")
        if auth != f"Bearer {API_TOKEN}":
            self._send_json(401, {"error": "Unauthorized"})
            return

        if self.path != "/v1/chat/completions":
            self._send_json(404, {"error": "Use /v1/chat/completions"})
            return

        length = int(self.headers.get("Content-Length", "0"))
        req_body = json.loads(self.rfile.read(length).decode("utf-8"))

        options = {
            "temperature": float(req_body.get("temperature", 0.1)),
            "num_predict": int(req_body.get("max_tokens", 3000)),
            "num_ctx": NUM_CTX,
        }
        if req_body.get("stop"):
            options["stop"] = req_body["stop"]

        ollama_payload = {
            "model": req_body.get("model") or MODEL_NAME,
            "messages": req_body.get("messages", []),
            "stream": False,
            "think": THINK_MODE,
            "options": options,
        }

        try:
            req = urllib.request.Request(
                "http://127.0.0.1:11434/api/chat",
                data=json.dumps(ollama_payload).encode("utf-8"),
                headers={"Content-Type": "application/json"},
            )
            with urllib.request.urlopen(req, timeout=300) as resp:
                ol = json.loads(resp.read().decode("utf-8"))
            content = (ol.get("message") or {}).get("content", "")
            self._send_json(
                200,
                {
                    "id": f"chatcmpl-colab-{int(time.time())}",
                    "object": "chat.completion",
                    "model": ollama_payload["model"],
                    "choices": [
                        {
                            "index": 0,
                            "message": {"role": "assistant", "content": content},
                            "finish_reason": ol.get("done_reason", "stop"),
                        }
                    ],
                    "usage": {
                        "prompt_tokens": ol.get("prompt_eval_count", 0),
                        "completion_tokens": ol.get("eval_count", 0),
                    },
                },
            )
        except Exception as exc:
            self._send_json(502, {"error": str(exc)})


proxy_server = ReusableThreadingHTTPServer(("127.0.0.1", PROXY_PORT), ColabOpenAIProxyHandler)
threading.Thread(target=proxy_server.serve_forever, daemon=True).start()

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
url_event = threading.Event()

def _drain_tunnel_logs():
    global public_url
    with open("/tmp/cloudflared.log", "w") as log_f:
        for line in tunnel_proc.stdout:
            log_f.write(line)
            log_f.flush()
            if public_url is None:
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                if match:
                    public_url = match.group(0)
                    url_event.set()

threading.Thread(target=_drain_tunnel_logs, daemon=True).start()
if not url_event.wait(timeout=45) or not public_url:
    raise RuntimeError("Could not obtain Cloudflare quick tunnel URL. Check /tmp/cloudflared.log")

print("\n======================================================")
print("Copy these lines into legal-copilot-backend/.env:")
print("======================================================")
print(f"COLAB_LLM_BASE_URL={public_url}")
print(f"COLAB_LLM_MODEL={MODEL_NAME}")
print(f"COLAB_LLM_API_KEY={API_TOKEN}")
print("COLAB_LLM_FALLBACK=false")
print("======================================================\n")
print("Run 3-model Colab benchmark locally (PowerShell):")
print("  .\\venv\\Scripts\\python.exe scripts\\benchmark\\run_benchmark.py --models colab_gemma4_e4b,colab_qwen3_8b,colab_llama3_1_8b --suites rag,contract,hallucination --quick")
print("Keep this Colab tab open while running local backend tests.")